In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Sheet1"
feature_cols = ["x1", "x2"]
target_col = "y"

df = pd.read_excel(file_path, sheet_name=sheet_name)
X_train = df[feature_cols].to_numpy(dtype=float)
y_train = df[target_col].to_numpy(dtype=float)
X_future = X_train[-5:, :]  # 示例

# ========= 2) 参数模板 =========
params = {
    "length_scale": 1.0,     # float: RBF长度尺度
    "constant_value": 1.0,   # float: 常数核系数
    "alpha": 1e-6            # float: 噪声项
}

kernel = C(params["constant_value"]) * RBF(length_scale=params["length_scale"])
gpr = GaussianProcessRegressor(kernel=kernel, alpha=params["alpha"], normalize_y=True)
gpr.fit(X_train, y_train)
mean_pred, std_pred = gpr.predict(X_future, return_std=True)
print("均值预测:", mean_pred)
print("标准差:", std_pred)


In [ ]:
"""
高斯回归预测模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "高斯回归预测模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "y"  # TODO: 请填写[因变量列名]，说明：连续数值列；其他数值列默认作为特征。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # 取出自变量和因变量；请保证目标列为连续数值。
    feature_cols = [c for c in data.columns if c != TARGET_COLUMN]
    X = data[feature_cols].to_numpy(dtype=float)
    y = data[TARGET_COLUMN].to_numpy(dtype=float)

    # 标准化能让核函数距离计算更稳定。
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # RBF 核控制平滑程度，WhiteKernel 描述观测噪声。
    kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0)
    model = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=RANDOM_STATE)
    model.fit(X_scaled, y)

    y_mean, y_std = model.predict(X_scaled, return_std=True)
    result = data.copy()
    result["预测值"] = y_mean
    result["预测标准差"] = y_std
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result.head())


if __name__ == "__main__":
    df = load_data()
    run_model(df)
